In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=100, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([90, 10]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [4]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 100,
    "multi_class": "auto",
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.93      1.00      0.96        27
           1       1.00      0.33      0.50         3

    accuracy                           0.93        30
   macro avg       0.97      0.67      0.73        30
weighted avg       0.94      0.93      0.92        30



In [5]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_dict

{'0': {'precision': 0.9310344827586207,
  'recall': 1.0,
  'f1-score': 0.9642857142857143,
  'support': 27.0},
 '1': {'precision': 1.0,
  'recall': 0.3333333333333333,
  'f1-score': 0.5,
  'support': 3.0},
 'accuracy': 0.9333333333333333,
 'macro avg': {'precision': 0.9655172413793103,
  'recall': 0.6666666666666666,
  'f1-score': 0.7321428571428572,
  'support': 30.0},
 'weighted avg': {'precision': 0.9379310344827586,
  'recall': 0.9333333333333333,
  'f1-score': 0.9178571428571428,
  'support': 30.0}}

In [6]:
import mlflow

In [7]:
mlflow.set_experiment("Rakesh")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5002/")

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy': report_dict['accuracy'],
        'recall_class_0': report_dict['0']['recall'],
        'recall_class_1': report_dict['1']['recall'],
        'f1_score_macro': report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(lr, "Logistic Regression")  

2025/09/09 23:14:30 INFO mlflow.tracking.fluent: Experiment with name 'Rakesh' does not exist. Creating a new experiment.
2025/09/09 23:14:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/09 23:14:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
